In [2]:
import numpy as np

In [ ]:
#### ARC grid for all metrics (no hybrid) ####

folder_file_to_read = 'results_PGM/accaccs/'
# folder_file_to_read = 'robustness_vs_uncertainty_results/auc'


# data_set_name = 'Australian'
# data_set_name = 'german'
# data_set_name = 'Healthy_aging'
# data_set_name = 'solar_big'
# data_set_name = 'solar_small'
# data_set_name = 'spect_heart'
# data_set_name = 'student_port_bin'
data_set_name = 'student_math_bin'



random_number = '_37'
sizes = ['_ALL']

noise = '_0.0'

model = '_NBC'
# model = '_GeFs'

file_extension = '.dat'


uncertainty_metrics = ['max_prob', 'entropies', 'aleatoric', 'epistemic', 'total', 'margin_of_confidence']
robustness_metrics = ['local_eps', 'total_var', 'chi2']

rob_metrics = {}

for size in sizes:
	file_to_read = folder_file_to_read + data_set_name + random_number + size + noise + model + file_extension

	with open(file_to_read, 'r') as f:
		lines = f.readlines()

		store_next_line = False

		for line in lines:
			if 'fixed' in line.strip():
				metric = line.split(':')[1].strip()
				store_next_line = True
				continue

			if store_next_line:
				rob_metrics[metric] = [x for x in line.strip().split(',')]

				store_next_line = False


	folder_file_to_write = '../../../../PGM26/paper/ARCs/csvs/'

	filename = folder_file_to_write + data_set_name + size + model + '_arc.csv'

	metrics = uncertainty_metrics + robustness_metrics

	with open(filename, 'w') as f:
		f.write('n'+','+','.join(metrics)+'\n')

		for i in range(len(rob_metrics['local_eps'])):
			line = str(i)
			for metric in metrics:
				line += ',' + rob_metrics[metric][i]
			f.write(line+'\n')

In [ ]:
#### ARCs per rejection rate ####

folder_file_to_read = 'results_PGM/accaccs/'
# folder_file_to_read = 'robustness_vs_uncertainty_results/auc'

data_set_names = ['adult',
					'Australian',
					'bank',
					'BCW',              # IGNORE: 99% accuracy, not a realistic use case for reliability quantification
					'german',
					# 'heart_disease',
					# 'lymphography',
					'Healthy_aging',
					# 'nursery',		# IGNORE: way too big and class imbalance leads to errors
					'solar_big',
					'solar_small',
					'spect_heart',
					'student_math_bin',
					'student_port_bin']



random_number = '_0'
second_random_number = '_1'

# numbers = [random_number]
numbers = [random_number, second_random_number]

sizes = ['_ALL']

noise = '_0.0'

model = '_NBC'
# model = '_GeFs'

file_extension = '.dat'


uncertainty_metrics = ['max_prob', 'entropies', 'aleatoric', 'epistemic', 'total', 'margin_of_confidence']
robustness_metrics = ['local_eps', 'total_var', 'chi2']

for data_set_name in data_set_names:
	rob_metrics_temp = {}

	for size in sizes:
		for r, random_num in enumerate(numbers):
			file_to_read = folder_file_to_read + data_set_name + random_num + model + file_extension

			with open(file_to_read, 'r') as f:
				lines = f.readlines()

				store_next_line = False

				for line in lines:
					if 'fixed' in line.strip():
						metric = line.split(':')[1].strip()
						store_next_line = True
						continue

					if store_next_line:
						if r == 0:
							rob_metrics_temp[metric] = []

						rob_metrics_temp[metric].append((np.array([float(x) for x in line.strip().split(',')])))

						store_next_line = False

		rob_metrics_temp = {key: np.array(rob_metrics_temp[key]) for key in rob_metrics_temp.keys()}

		rob_metrics = {}

		for key in rob_metrics_temp.keys():
			rob_metrics[key] = np.mean(rob_metrics_temp[key], axis=0)


		folder_file_to_write = 'results_PGM/latex_plots/ARCs/csvs/'

		filename = folder_file_to_write + data_set_name + size + model + '_arc.csv'

		metrics = uncertainty_metrics + robustness_metrics

		with open(filename, 'w') as f:
			f.write('n'+','+','.join(metrics)+'\n')

			for i in range(len(rob_metrics['local_eps'])):
				line = str(i)
				for metric in metrics:
					line += ',' + str(rob_metrics[metric][i])
				f.write(line+'\n')

In [ ]:
#### ARC values at certain rejection rates ####

folder_file_to_read = 'results_PGM/accaccs/'

data_set_names = ['adult',
					'Australian',
					'bank',
					'BCW',              # IGNORE: 99% accuracy, not a realistic use case for reliability quantification
					'german',
					# 'heart_disease',
					# 'lymphography',
					'Healthy_aging',
					# 'nursery',		# IGNORE: way too big and class imbalance leads to errors
					'solar_big',
					'solar_small',
					'spect_heart',
					'student_math_bin',
					'student_port_bin']

# ready = ['solar_small', 'Australian', 'student_math_bin', 'student_port_bin', 'german', 'solar_big', 'Healthy_aging', 'bank', 'adult', 'BCW', 'spect_heart']

# data_set_name = 'Australian'
# data_set_name = 'german'
# data_set_name = 'Healthy_aging'
# data_set_name = 'solar_big'
# data_set_name = 'solar_small'
# data_set_name = 'spect_heart'
# data_set_name = 'student_port_bin'
# data_set_name = 'student_math_bin'
# data_set_name = 'BCW'
# data_set_name = 'bank'

rejection_rates = [0.1, 0.25, 0.75, 0.9]


random_number = '_0'
second_random_number = '_1'

numbers = [random_number]
# numbers = [random_number, second_random_number]

sizes = ['_ALL']

noise = '_0.0'

model = '_NBC'
# model = '_GeFs'

file_extension = '.dat'

uncertainty_metrics = ['total', 'aleatoric', 'epistemic', 'max_prob', 'entropies', 'margin_of_confidence']
robustness_metrics = ['chi2', 'total_var', 'local_eps']
metrics = np.array(robustness_metrics + uncertainty_metrics)

maxes_at_rr = [[] for _ in range(len(rejection_rates))]


for d_i, data_set_name in enumerate(data_set_names):
	rob_metrics_temp = {}
	# rob_metrics_A = {}
	# rob_metrics_B = {}

	for size in sizes:
		for r, random_num in enumerate(numbers):

			file_to_read = folder_file_to_read + data_set_name + random_num + model + file_extension

			with open(file_to_read, 'r') as f:
				lines = f.readlines()

				store_next_line = False

				for line in lines:
					if 'fixed' in line.strip():
						metric = line.split(':')[1].strip()
						store_next_line = True
						continue

					if store_next_line:
						if r == 0:
							rob_metrics_temp[metric] = []

						rob_metrics_temp[metric].append((np.array([float(x) for x in line.strip().split(',')])))

						store_next_line = False

		rob_metrics_temp = {key: np.array(rob_metrics_temp[key]) for key in rob_metrics_temp.keys()}

		rob_metrics = {}

		for key in rob_metrics_temp.keys():
			rob_metrics[key] = np.mean(rob_metrics_temp[key], axis=0)


		N = len(rob_metrics['local_eps'])
		rejection_rate_indices = [int(N * rr) for rr in rejection_rates]
		# print(rejection_rate_indices)

		for i, rr in enumerate(rejection_rates):
			filename = f"ARCs_at_{rr:.2f}_rejection_rate.csv"

			if d_i == 0:
				with open(filename, 'w') as f:
					f.write('dataset'+','+','.join(metrics)+'\n')

			with open(filename, 'a') as f:
				line = data_set_name
				values_at_rr = []
				for metric in metrics:
					values_at_rr.append(rob_metrics[metric][rejection_rate_indices[i]])
					line += ',' + f"{rob_metrics[metric][rejection_rate_indices[i]]:.4f}"
				f.write(line+'\n')

				max_indices = np.where(values_at_rr == np.max(values_at_rr))[0]

				maxes_at_rr[i]+= list(metrics[max_indices])



folder_file_to_write = '../../../../PGM26/paper/RR_ARC/csvs/'

filename = folder_file_to_write + f"table_rej_rates{model}.csv"

with open(filename, 'w') as f:
	f.write('dataset'+','+','.join(metrics)+'\n')

for i, rr in enumerate(rejection_rates):
	# print(f"Rejection rate: {rr:.2f}")
	# count number of occurences of each metric in maxes_at_rr[i]
	maxes_at_rr[i] = np.array(maxes_at_rr[i])
	uniques = np.unique(maxes_at_rr[i], return_counts=True)
	max_metrics = uniques[0]
	number_of_wins = uniques[1]


	with open(filename, 'a') as f:
		line = f"{rr*100:.0f}\%"
		for metric in metrics:
			if metric in max_metrics:
				line += f",{number_of_wins[np.where(max_metrics == metric)][0]}"
			else:
				line += f",0"
		f.write(line+'\n')



	# zip the unique metrics and their counts into a dictionary
	metric_counts = list(zip(uniques[0], uniques[1]))
	sorted_metric_counts = sorted(metric_counts, key=lambda x: x[1], reverse=True)
	# print(sorted_metric_counts, '\n')

<>:157: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
<>:157: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
/var/folders/vg/cwp2f4q11xs6h9l57j045fw40000gn/T/ipykernel_51823/1801627108.py:157: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
  line = f"{rr*100:.0f}\%"


In [203]:
#### mean ARC values per rejection rate ####

folder_file_to_read = 'results_PGM/accaccs/'
# folder_file_to_read = 'robustness_vs_uncertainty_results/auc'

data_set_names = ['adult',
					'Australian',
					'bank',
					'BCW',              # IGNORE: 99% accuracy, not a realistic use case for reliability quantification
					'german',
					# 'heart_disease',
					# 'lymphography',
					'Healthy_aging',
					# 'nursery',		# IGNORE: way too big and class imbalance leads to errors
					'solar_big',
					'solar_small',
					'spect_heart',
					'student_math_bin',
					'student_port_bin']

ready = ['solar_small', 'Australian', 'student_math_bin', 'student_port_bin', 'german', 'solar_big', 'Healthy_aging', 'bank']
# data_set_name = 'bank'

rejection_rates = [0.1, 0.25, 0.75, 0.9]
# rejection_rates = [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]


random_number = '_288'
second_random_number = '_629'

# numbers = [random_number]
numbers = [random_number, second_random_number]

sizes = ['_ALL']

noise = '_0.0'

# model = '_NBC'
model = '_GeFs'

file_extension = '.dat'

uncertainty_metrics = ['total', 'aleatoric', 'epistemic', 'max_prob', 'entropies', 'margin_of_confidence']
robustness_metrics = ['chi2', 'total_var', 'local_eps']
metrics = np.array(robustness_metrics + uncertainty_metrics)

mean_at_rr = [[] for _ in range(len(rejection_rates))]


for d_i, data_set_name in enumerate(data_set_names):
	rob_metrics_temp = {}

	for size in sizes:
		for r, random_num in enumerate(numbers):
			if r > 0 and data_set_name not in ready:
				continue

			file_to_read = folder_file_to_read + data_set_name + random_num + size + noise + model + file_extension

			with open(file_to_read, 'r') as f:
				lines = f.readlines()

				store_next_line = False

				for line in lines:
					if 'fixed' in line.strip():
						metric = line.split(':')[1].strip()
						store_next_line = True
						continue

					if store_next_line:
						if r == 0:
							rob_metrics_temp[metric] = []

						rob_metrics_temp[metric].append((np.array([float(x) for x in line.strip().split(',')])))

						store_next_line = False

		rob_metrics_temp = {key: np.array(rob_metrics_temp[key]) for key in rob_metrics_temp.keys()}

		rob_metrics = {}

		for key in rob_metrics_temp.keys():
			rob_metrics[key] = np.mean(rob_metrics_temp[key], axis=0)


		N = len(rob_metrics['local_eps'])
		rejection_rate_indices = [int(N * rr) for rr in rejection_rates]
		# print(rejection_rate_indices)

		for i, rr in enumerate(rejection_rates):
			filename = f"ARCs_at_{rr:.2f}_rejection_rate.csv"

			if d_i == 0:
				with open(filename, 'w') as f:
					f.write('dataset'+','+','.join(metrics)+'\n')

			with open(filename, 'a') as f:
				line = data_set_name
				values_at_rr = []
				for metric in metrics:
					values_at_rr.append(rob_metrics[metric][rejection_rate_indices[i]])
					line += ',' + f"{rob_metrics[metric][rejection_rate_indices[i]]:.4f}"
				f.write(line+'\n')
				values_at_rr = np.array(values_at_rr)
				# print(values_at_rr)

				mean_at_rr[i].append(values_at_rr)

				# max_indices = np.where(values_at_rr == np.max(values_at_rr))[0]

				# maxes_at_rr[i]+= list(metrics[max_indices])

		# folder_file_to_write = '../../../../PGM26/paper/ARCs/csvs/'

		# filename = folder_file_to_write + data_set_name + size + model + '_arc.csv'

		# filename = f"{}"

		# metrics = uncertainty_metrics + robustness_metrics

		# with open(filename, 'w') as f:
		# 	f.write('n'+','+','.join(metrics)+'\n')

		# 	for i in rejection_rate_indices:
		# 		line = str(i)
		# 		for metric in metrics:
		# 			line += ',' + f"{rob_metrics[metric][i]:.4f}"
		# 		f.write(line+'\n')
# print(mean_at_rr)
mean_at_rr = np.array(mean_at_rr)
# take mean over datasets for each metric at each rejection rate
mean_at_rr = np.mean(mean_at_rr, axis=1)
# print(mean_at_rr.shape)


folder_file_to_write = '../../../../PGM26/paper/RR_ARC/csvs/'

filename = folder_file_to_write + f"table_rr_mean_arcs{model}.csv"

with open(filename, 'w') as f:
	f.write('dataset'+','+','.join(metrics)+'\n')

for i, rr in enumerate(rejection_rates):
	# print(f"Rejection rate: {rr:.2f}")
	# count number of occurences of each metric in maxes_at_rr[i]
	# maxes_at_rr[i] = np.array(maxes_at_rr[i])
	# uniques = np.unique(maxes_at_rr[i], return_counts=True)
	# max_metrics = uniques[0]
	# number_of_wins = uniques[1]


	with open(filename, 'a') as f:
		line = f"{rr*100:.0f}\%"
		for j, metric in enumerate(metrics):
			line += f",{mean_at_rr[i][j]:.4f}"
			# if metric in max_metrics:
			# 	line += f",{number_of_wins[np.where(max_metrics == metric)][0]}"
			# else:
			# 	line += f",0"
		f.write(line+'\n')


	# zip the unique metrics and their counts into a dictionary
	metric_counts = list(zip(uniques[0], uniques[1]))
	sorted_metric_counts = sorted(metric_counts, key=lambda x: x[1], reverse=True)
	# print(sorted_metric_counts, '\n')

<>:154: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
<>:154: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
/var/folders/vg/cwp2f4q11xs6h9l57j045fw40000gn/T/ipykernel_51823/1796657066.py:154: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
  line = f"{rr*100:.0f}\%"


In [23]:
#### combi mean and wins ARC values per rejection rate ####

folder_file_to_read = 'results_PGM/accaccs/'
# folder_file_to_read = 'robustness_vs_uncertainty_results/auc'

data_set_names = ['adult',
					'Australian',
					'bank',
					'BCW',              # IGNORE: 99% accuracy, not a realistic use case for reliability quantification
					'german',
					# 'heart_disease',
					# 'lymphography',
					'Healthy_aging',
					# 'nursery',		# IGNORE: way too big and class imbalance leads to errors
					'solar_big',
					'solar_small',
					'spect_heart',
					'student_math_bin',
					'student_port_bin']

# ready = ['solar_small', 'Australian', 'student_math_bin', 'student_port_bin', 'german', 'solar_big', 'Healthy_aging', 'bank', 'adult', 'BCW']
# data_set_name = 'bank'

rejection_rates = [0.1, 0.25, 0.75, 0.9]
# rejection_rates = [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]


random_number = '_0'
second_random_number = '_1'

# numbers = [random_number]
numbers = [random_number, second_random_number]

sizes = ['_ALL']

# model = '_NBC'
model = '_GeFs'

file_extension = '.dat'

uncertainty_metrics = ['total', 'aleatoric', 'epistemic', 'max_prob', 'entropies', 'margin_of_confidence']
robustness_metrics = ['chi2', 'total_var', 'local_eps']
metrics = np.array(robustness_metrics + uncertainty_metrics)

mean_at_rr = [[] for _ in range(len(rejection_rates))]
maxes_at_rr = [[] for _ in range(len(rejection_rates))]

for d_i, data_set_name in enumerate(data_set_names):
	rob_metrics_temp = {}

	for size in sizes:
		for r, random_num in enumerate(numbers):

			file_to_read = folder_file_to_read + data_set_name + random_num + model + file_extension

			with open(file_to_read, 'r') as f:
				lines = f.readlines()

				store_next_line = False

				for line in lines:
					if 'fixed' in line.strip():
						metric = line.split(':')[1].strip()
						store_next_line = True
						continue

					if store_next_line:
						if r == 0:
							rob_metrics_temp[metric] = []

						rob_metrics_temp[metric].append((np.array([float(x) for x in line.strip().split(',')])))

						store_next_line = False

		rob_metrics_temp = {key: np.array(rob_metrics_temp[key]) for key in rob_metrics_temp.keys()}

		rob_metrics = {}

		for key in rob_metrics_temp.keys():
			rob_metrics[key] = np.mean(rob_metrics_temp[key], axis=0)


		N = len(rob_metrics['local_eps'])
		rejection_rate_indices = [int(N * rr) for rr in rejection_rates]

		for i, rr in enumerate(rejection_rates):

			values_at_rr = []
			for metric in metrics:
				values_at_rr.append(rob_metrics[metric][rejection_rate_indices[i]])

			max_indices = np.where(values_at_rr == np.max(values_at_rr))[0]

			maxes_at_rr[i]+= list(metrics[max_indices])

			values_at_rr = np.array(values_at_rr)
			mean_at_rr[i].append(values_at_rr)


mean_at_rr = np.array(mean_at_rr)
mean_at_rr = np.mean(mean_at_rr, axis=1)


folder_file_to_write = 'results_PGM/latex_plots/RR_ARC/csvs/'

filename = folder_file_to_write + f"table_rr_combi{model}.csv"

with open(filename, 'w') as f:
	f.write('dataset'+','+',,'.join(metrics)+','+'\n')
	f.write(''+','+','.join(['\%', 'W']*len(metrics))+'\n')

for i, rr in enumerate(rejection_rates):
	# print(f"Rejection rate: {rr:.2f}")
	maxes_at_rr[i] = np.array(maxes_at_rr[i])
	uniques = np.unique(maxes_at_rr[i], return_counts=True)
	max_metrics = uniques[0]
	number_of_wins = uniques[1]


	with open(filename, 'a') as f:
		line = f"{rr*100:.0f}\%"
		for j, metric in enumerate(metrics):
			line += f",{mean_at_rr[i][j]*100:.2f}"

			if metric in max_metrics:
				line += f",{number_of_wins[np.where(max_metrics == metric)][0]}"
			else:
				line += f",0"

		f.write(line+'\n')


	# zip the unique metrics and their counts into a dictionary
	metric_counts = list(zip(uniques[0], uniques[1]))
	sorted_metric_counts = sorted(metric_counts, key=lambda x: x[1], reverse=True)
	# print(sorted_metric_counts, '\n')

<>:110: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
<>:121: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
<>:110: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
<>:121: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
/var/folders/vg/cwp2f4q11xs6h9l57j045fw40000gn/T/ipykernel_86040/3060104225.py:110: SyntaxWarning: "\%" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\%"? A raw string is also an option.
  f.write(''+','+','.join(['\%', 'W']*len(metrics))+'\n')
/var/folders/vg/cwp2f4q11xs6h9l57j045fw40000gn/T/ipykernel_86040/3060104225.py:121: SyntaxW

In [ ]:
#### AUC TABLE - all metrics, no hybrids ####
write_aucs_for_rr_table = True

folder_file_to_read = 'results_PGM/auc_summary/'
# folder_file_to_read = 'results_DS/auc/'

data_set_names = ['adult',
					'Australian',
					'bank',
					'BCW',              # IGNORE: 99% accuracy, not a realistic use case for reliability quantification
					'german',
					# 'heart_disease',
					# 'lymphography',
					'Healthy_aging',
					# 'nursery',		# IGNORE: way too big and class imbalance leads to errors
					'solar_big',
					'solar_small',
					'spect_heart',
					'student_math_bin',
					'student_port_bin']

random_number = '0'
second_random_number = '1'

# numbers = [random_number]
numbers = [random_number, second_random_number]

nbc_model = False

if nbc_model:
	model_architecture = '_NBC'
else:
	model_architecture = '_GeFs'



file_extension = '.dat'

data_set_dicts = {}

uncertainty_metrics = ['total', 'aleatori', 'epistemi', 'max_prob', 'entropie', 'margin_o']
robustness_metrics = ['chi2', 'total_va', 'local_ep']

list_of_metrics =robustness_metrics + uncertainty_metrics

precision = 5


datasets = {}

aucs_per_metric = {}

for i, random_num in enumerate(numbers):
	file_to_read = folder_file_to_read + random_num + model_architecture + file_extension

	with open(file_to_read, 'r') as f:
		lines = f.readlines()

		for line in lines[2:]:
			if line == '\n':
				continue
			setn = line[:10].strip()
			# aucs = []
			aucs = {}

			metrics = line.split(':')[1].strip().split('|')

			best_unc = False
			for metric in metrics:
				metric_name = metric.split('(')[0].strip()

				if metric_name not in aucs_per_metric.keys():
					aucs_per_metric[metric_name] = []

				# print(metric_name[:8])

				if metric_name[:8] in robustness_metrics:
					auc = metric.split('(')[1].split("±")[0].strip()
					# aucs.append((metric_name, auc))
					if i == 0:
						aucs[metric_name] = float(auc)
					else:
						datasets[setn][metric_name] += float(auc)
						datasets[setn][metric_name] /= 2
					aucs_per_metric[metric_name].append(float(auc))

				elif not best_unc:
					auc = metric.split('(')[1].split("±")[0].strip()
					# aucs.append((metric_name, auc))
					best_unc = True
					if i == 0:
						aucs['uncertainty'] = float(auc)
					else:
						# print(datasets)
						datasets[setn]['uncertainty'] += float(auc)
						datasets[setn]['uncertainty'] /= 2
					aucs_per_metric[metric_name].append(float(auc))

				else:
					auc = metric.split('(')[1].split("±")[0].strip()
					# aucs.append((metric_name, auc))
					# best_unc = True
					# aucs['uncertainty'] = float(auc)
					aucs_per_metric[metric_name].append(float(auc))
					continue

			# print(metrics)
			if i == 0:
				datasets[setn] = aucs
			# print(line[:10].strip())

mean_aucs_per_metric = {m: np.mean(aucs_per_metric[m]) for m in aucs_per_metric.keys()}
# print(mean_aucs_per_metric)

mean_aucs = [(mean_aucs_per_metric[m],m) for m in aucs_per_metric.keys()]

# sort list of tuples by mean auc, first element of tuple, in descending order
mean_aucs.sort(reverse=True)

if write_aucs_for_rr_table:
	rejection_rates = [0.1, 0.25, 0.75, 0.9]
	folder_file_to_write = 'results_PGM/latex_plots/RR_ARC/csvs/'

	filename = folder_file_to_write + f"table_rr_combi{model_architecture}.csv"

	with open(filename, 'a') as f:
		line = f"{{\\footnotesize AU-ARC}}"
		for metric in list_of_metrics:
			# make metric max 8 characters long
			metric_short = metric[:8]
			line += f",\\SetCell[c=2]{{c}} {mean_aucs_per_metric[metric]:.5f},"
		f.write(line+'\n')

		# zip the unique metrics and their counts into a dictionary
		metric_counts = list(zip(uniques[0], uniques[1]))
		sorted_metric_counts = sorted(metric_counts, key=lambda x: x[1], reverse=True)
	# print(sorted_metric_counts, '\n')

else:
	folder_file_to_write = 'results_PGM/latex_plots/AUCs/csvs/'


	# uncertainty_metrics = ['margin', 'maxprob', 'aleatoric', 'total', 'epistemic', 'entropies']
	# robustness_metrics = ['robglobal', 'roblocal']

	# metrics = uncertainty_metrics + robustness_metrics

	header = ['dataset', 'chisquared', 'TV', 'local_eps', 'uncertainty']

	file_to_write = folder_file_to_write + 'table_' + random_number + set_name_suffix + model_architecture + '.csv'

	with open(file_to_write, 'w') as f:
		# write header line with all metrics separated by commas
		f.write(','.join(header)+'\n')

		for i, data_set_name in enumerate(data_set_names):

			dsn = 'D' + f"{i+1}"
			aucs = datasets[data_set_name[:10]]

			# dsn = dsn[0].upper() + dsn[1:]

			f.write(dsn + ',' +
						f'{aucs['chi2']:.4f}' + ',' +
						f'{aucs['total_va']:.4f}' + ',' +
						f'{aucs['local_ep']:.4f}' + ',' +
						f'{aucs['uncertainty']:.4f}'
						+'\n')

{'chi2': np.float64(0.8795172727272729), 'total_va': np.float64(0.8800254545454546), 'local_ep': np.float64(0.8780990909090908), 'epistemi': np.float64(0.8773563636363637), 'total': np.float64(0.8707790909090908), 'aleatori': np.float64(0.8614936363636363), 'max_prob': np.float64(0.8415586363636365), 'entropie': np.float64(0.841395), 'margin_o': np.float64(0.8416045454545454)}
